# Lab 012 — Random Forest vs a single tree

**Lesson:** [`lessons/0012-bagging-random-forest.html`](../lessons/0012-bagging-random-forest.html) · **Phase / Year:** Year 1 · Q2

**Dataset tier:** A — OpenML German credit (`credit_g`) via `relkit` (same harness as Lab 011).

**Skill you are practising:** score a single deep tree and a Random Forest with the shared CV harness, read the OOB estimate, and measure the variance drop that bagging buys.

**Exit criteria:** EXIT TICKET prints single-tree vs RF CV PR-AUC, the RF OOB score, and the per-row prediction-variance drop from averaging.

---

### How this notebook works
- **PROVIDED** cells — complete boilerplate; just run.
- **TODO** cells — blanks (`____` / `# TODO`); you implement the skill.
- **CHECK** cells — immediate feedback; do not edit.
- Run top to bottom. When **EXIT TICKET** prints cleanly, paste it to your teacher or say *"lab done"*.

### Environment
One-time: `bash labs/setup-env.sh` from repo root → kernel **Relational Labs (.venv)**. No local install? Open this notebook from [`notebooks.html`](../notebooks.html) — **View** to read it rendered, or **Run on Binder** to execute it in the browser.

### Running on Google Colab?

Colab opens only this single file, so the course package (`relkit`) and the lab
dependencies (xgboost, lightgbm, catboost, …) are **not** present by default. The cell
below fixes that: on Colab it shallow-clones the course repo, installs
`requirements-labs.txt`, and switches into `labs/` so `relkit` imports and the data cache
resolve. **On a local venv or Binder it does nothing — just run it and continue.**

In [ ]:
# @colab-bootstrap — PROVIDED. Makes the lab self-sufficient on Google Colab; a no-op elsewhere.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/relational"):
        !git clone --depth 1 https://github.com/Avistian/relational.git /content/relational
    %pip install -q -r /content/relational/requirements-labs.txt
    os.chdir("/content/relational/labs")
    print("Colab ready — working dir:", os.getcwd())
else:
    print("Not on Colab — using the local environment as-is.")

## Concept recap — bagging and Random Forest

A **deep decision tree** (Lesson 011) has **low bias but high variance**: it fits the training rows almost exactly, but resampling a few rows shifts its splits a lot. Two independent ways to combine trees fix two different problems:

- **Bagging** (*bootstrap aggregating*) — train `B` trees, each on a **bootstrap sample** (draw `n` rows *with replacement*), then **average** their probabilities. If the trees were independent, variance falls by about `1/B` while bias stays put.
- **Random Forest** — bagging **plus** a random **feature subset at every split** (often `√p`). This *decorrelates* the trees, so the average reduces variance more than plain bagging.

**Out-of-bag (OOB):** a bootstrap leaves out ~37% of rows (`(1−1/n)^n → e^{-1}`). Scoring each row on the trees that never saw it gives a nearly-free validation estimate.

**Why it matters for the thesis:** RF is a second honest baseline. But boosting (Q1 HistGBDT, XGBoost in L014) usually *beats* RF on tabular because it cuts **bias**, not just variance — RF is the control that isolates *what* the later win is really from.

**Toy intuition (not this dataset):** three noisy estimates of a true 0.60 probability — `0.9, 0.3, 0.6` — average to `0.60`. Each one alone is far off; the mean is close. That is the whole trick, scaled to hundreds of trees.

Full viz (variance collapsing as `B` grows) and reading: [Lesson 012](../lessons/0012-bagging-random-forest.html) · Breiman 2001.

## Setup — PROVIDED

In [ ]:
# PROVIDED
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))  # labs/ on the path so `relkit` imports

import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from relkit.data import load_tier_a
from relkit.metrics import cv_pr_auc

RS = 0
print("relkit ok")

## Data — PROVIDED (Tier A)

Same German credit data as Lab 011: 1000 rows, positive rate ~0.70. Categoricals are label-encoded to integers **for this demo** so bare tree estimators accept them (production baselines would use a `ColumnTransformer` pipeline, Lesson 005).

Run to load and encode.

In [ ]:
# PROVIDED
X, y = load_tier_a("credit_g")
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

X_num = X[num_cols].copy()
for c in cat_cols:
    X_num[c] = LabelEncoder().fit_transform(X[c].astype(str))

y = np.asarray(y)
print(f"rows={len(y)}  pos_rate={y.mean():.3f}  features={X_num.shape[1]}")

## Task 1 — Single-tree baseline — TODO

**Goal:** the 5-fold CV PR-AUC of one deep, unpruned tree.

**Why it matters:** this is the high-variance learner bagging is meant to stabilize — you need its score to prove the forest actually helps.

**You implement:**
1. `DecisionTreeClassifier(random_state=RS)` — no `max_depth` (grow it fully).
2. `cv_pr_auc(estimator, X_num, y)` → `tree_pr`.

In [ ]:
# TODO
tree = ____              # deep DecisionTreeClassifier(random_state=RS)
tree_pr = ____           # cv_pr_auc(tree, X_num, y)
print(f"single deep tree  CV PR-AUC = {tree_pr:.3f}")

In [ ]:
# CHECK — do not edit
assert 0.0 <= tree_pr <= 1.0, "PR-AUC must be in [0, 1]."
assert tree_pr > y.mean() - 0.05, "A tree should be near or above the prevalence floor."
print("Task 1 ok — single-tree baseline scored.")

## Task 2 — Random Forest + OOB — TODO

**Goal:** the CV PR-AUC of a 300-tree Random Forest, plus its out-of-bag score.

**Why it matters:** the forest averages decorrelated trees — you should see PR-AUC jump well above the single tree. OOB gives a near-free validation estimate.

**You implement:**
1. `RandomForestClassifier(n_estimators=300, random_state=RS, n_jobs=-1, oob_score=True)`.
2. `cv_pr_auc(...)` → `rf_pr`.
3. `.fit(X_num, y)` then read `.oob_score_` → `oob`.

In [ ]:
# TODO
rf = ____                # RandomForestClassifier(n_estimators=300, random_state=RS, n_jobs=-1, oob_score=True)
rf_pr = ____             # cv_pr_auc(rf, X_num, y)
____                     # fit rf on the full data so oob_score_ is available
oob = ____               # rf.oob_score_
print(f"random forest 300 CV PR-AUC = {rf_pr:.3f}   OOB accuracy = {oob:.3f}")

In [ ]:
# CHECK — do not edit
assert 0.0 <= rf_pr <= 1.0 and 0.0 <= oob <= 1.0
assert rf_pr > tree_pr, f"RF ({rf_pr:.3f}) should beat the single tree ({tree_pr:.3f})."
print("Task 2 ok — forest beats the single tree.")

## Task 3 — Measure the variance drop — TODO (crucial fragment)

**Goal:** show averaging shrinks variance. Fit many single trees on different bootstraps, look at how much their predictions for a fixed set of rows wobble, then compare a 10-tree average.

**Why it matters:** "variance reduction" is abstract until you watch the spread of predictions collapse — this is bagging's whole justification, and the `~1/B` law in action.

**You implement (loop is PROVIDED except the blanks):**
1. For each bootstrap, fit a deep tree on the resampled rows and predict `proba[:,1]` for the fixed rows.
2. `single_std` = mean over rows of the std across the 50 trees' predictions.
3. `ens_std` = mean over rows of the std across four **10-tree averages**.

**Expect:** `ens_std` roughly a third to a quarter of `single_std`.

In [ ]:
# PARTLY PROVIDED — fill the blanks
rng = np.random.RandomState(0)
n = len(y)
fixed = np.arange(20)                       # 20 fixed rows we keep re-predicting
preds = []
for b in range(50):
    bs = rng.randint(0, n, n)               # a bootstrap sample of row indices
    t = DecisionTreeClassifier(random_state=b).fit(X_num.iloc[bs], y[bs])
    preds.append(t.predict_proba(X_num.iloc[fixed])[:, 1])
preds = np.array(preds)                     # shape (50 trees, 20 rows)

single_std = ____                           # preds.std(axis=0).mean()
ens10 = np.array([preds[i:i+10].mean(axis=0) for i in range(0, 40, 10)])  # four 10-tree averages
ens_std = ____                              # ens10.std(axis=0).mean()
print(f"per-row std: single tree = {single_std:.3f}   10-tree ensemble = {ens_std:.3f}")

In [ ]:
# CHECK — do not edit
assert single_std > ens_std, "Averaging should reduce prediction variance."
assert ens_std < 0.6 * single_std, "Expected a clear variance drop from averaging."
print("Task 3 ok — variance shrinks with averaging.")

## Exit ticket — TODO

**Goal:** one printed summary to paste back to your teacher.

**Takeaway prompt:** in one sentence, say what bagging reduced (variance vs bias) and why the forest beat the single tree — and note whether you'd still expect a boosted GBDT to beat this RF.

In [ ]:
# TODO: complete the takeaway string
print("=== EXIT TICKET — Lesson 012 ===")
print(f"single tree CV PR-AUC : {tree_pr:.3f}")
print(f"random forest PR-AUC  : {rf_pr:.3f}   (OOB acc {oob:.3f})")
print(f"variance std          : single {single_std:.3f}  ->  10-tree {ens_std:.3f}")
print()
print("takeaway:", "____")